# LLM Classification Finetuning

## 1. Train

In [1]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import torch
import evaluate

LLM_MODEL = "distilbert-base-uncased"
MAX_TOKEN_LENGTH = 512
data_submission_path = "submission.csv"



/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


train_data_path = "kaggle/input/competitions/llm-classification-finetuning/train.csv"
test_data_path = "kaggle/input/competitions/llm-classification-finetuning/test.csv"
model_save_path = "./model"



In [3]:
def maybe_swap(row):
    if np.random.rand() < 0.5:
        row["response_a"], row["response_b"] = row["response_b"], row["response_a"]
        if row["labels"] == 0:
            row["labels"] = 1
        elif row["labels"] == 1:
            row["labels"] = 0
    return row

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_TOKEN_LENGTH,
    )

def truncate_text(text, max_tokens):
    ids = tokenizer(
        str(text),
        truncation=True,
        max_length=max_tokens,
        add_special_tokens=False
    )["input_ids"]
    return tokenizer.decode(ids)

def build_text(row):
    prompt = truncate_text(row["prompt"], 128)
    a = truncate_text(row["response_a"], 192)
    b = truncate_text(row["response_b"], 192)

    return (
        "[PROMPT]\n" + prompt +
        "\n\n[RESPONSE A]\n" + a +
        "\n\n[RESPONSE B]\n" + b
    )

In [4]:

# 0) Data loading 
df = pd.read_csv(train_data_path)

# 1) Label encoding
if set(["winner_model_a", "winner_model_b", "winner_tie"]).issubset(df.columns):
    df["labels"] = (
        df["winner_model_a"] * 0 +
        df["winner_model_b"] * 1 +
        df["winner_tie"] * 2
    ).astype("int64")
else:
    raise ValueError("Missing one of winner_model_a/winner_model_b/winner_tie")

# 2) Ramdomly swap response_a and response_b for half the data to prevent positional bias
df = df.apply(maybe_swap, axis=1)


# 3) Build text field
if set(["prompt", "response_a", "response_b"]).issubset(df.columns):
    df["text"] = df.apply(build_text, axis=1)
else:
    raise ValueError("Missing one of prompt/response_a/response_b")

print("Loaded", len(df), "rows")
print(df["labels"].value_counts())

Loaded 57477 rows
labels
0    19903
1    19813
2    17761
Name: count, dtype: int64


In [5]:


# 4) HF dataset
dataset = Dataset.from_pandas(df[["text", "labels"]])

# 5) split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

# 6) tokenize
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


Map: 100%|██████████| 5748/5748 [00:00<00:00, 9972.35 examples/s] 


In [6]:


model = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL,
    num_labels=3
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 17241.35it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


training_args = TrainingArguments(
    output_dir=model_save_path,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    max_steps=-1,
    warmup_steps=0.1,
    weight_decay=0.01,
    fp16=False,
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=120,
    save_steps=480,
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    logging_nan_inf_filter=False,
    # use_cpu=True,
)


In [8]:


data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics, 
)

In [9]:
#B. Check a manual forward pass on a train batch

# Do this before full training and then again after a few steps:

name, p = next((n, p) for n, p in model.named_parameters() if p.requires_grad)
before = p.detach().cpu().clone()

trainer.train()

after = dict(model.named_parameters())[name].detach().cpu()
print("param changed?", not torch.equal(before, after))
print("max abs diff:", (after - before).abs().max().item())

/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
120,4.399746,1.098899,0.340118
240,4.387852,1.094364,0.363257
360,4.398461,1.095066,0.346033
480,4.377204,1.090609,0.376653
600,4.332953,1.095298,0.359951
720,4.316870,1.098204,0.371955
840,4.330661,1.087334,0.397008
960,4.401032,1.094021,0.359081
1080,4.367538,1.091562,0.397356
1200,4.385773,1.091564,0.385525


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 13.87it/s]
/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, dev

param changed? True
max abs diff: 0.008078880608081818


In [10]:
trainer.remove_callback(type(trainer.callback_handler.callbacks[-1]))  # not ideal/generic

In [11]:


print("global_step:", trainer.state.global_step)
print(trainer.state.log_history[-10:])
print("val metrics:", trainer.evaluate(val_dataset))

pred = trainer.predict(val_dataset)
print("nan/inf", np.any(np.isnan(pred.predictions)), np.any(np.isinf(pred.predictions)))

/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


global_step: 12934
[{'loss': 4.137704467773437, 'grad_norm': 3.878843307495117, 'learning_rate': 3.178694158075601e-07, 'epoch': 1.9716605451382176, 'step': 12750}, {'loss': 4.141954040527343, 'grad_norm': 5.8523077964782715, 'learning_rate': 2.7491408934707903e-07, 'epoch': 1.9755267736323217, 'step': 12775}, {'loss': 4.420845947265625, 'grad_norm': 3.2337472438812256, 'learning_rate': 2.3195876288659797e-07, 'epoch': 1.9793930021264257, 'step': 12800}, {'loss': 4.176254577636719, 'grad_norm': 4.02688455581665, 'learning_rate': 1.8900343642611685e-07, 'epoch': 1.9832592306205297, 'step': 12825}, {'eval_loss': 1.044942855834961, 'eval_accuracy': 0.44467640918580376, 'eval_runtime': 60.4961, 'eval_samples_per_second': 95.014, 'eval_steps_per_second': 11.885, 'epoch': 1.9855789677169922, 'step': 12840}, {'loss': 4.1304190063476565, 'grad_norm': 3.1878628730773926, 'learning_rate': 1.4604810996563575e-07, 'epoch': 1.9871254591146337, 'step': 12850}, {'loss': 4.137420043945313, 'grad_norm'

/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


nan/inf False False


In [12]:
# Checking prediction distribution to see if model is learning anything or just predicting one class

pred = trainer.predict(val_dataset)
pred_labels = pred.predictions.argmax(axis=-1)

from collections import Counter
print("pred distribution:", Counter(pred_labels))
print("true distribution:", Counter(val_dataset["labels"]))



/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


pred distribution: Counter({np.int64(0): 2709, np.int64(1): 2010, np.int64(2): 1029})
true distribution: Counter({0: 2047, 1: 1942, 2: 1759})


In [13]:

# Inspect token length distribution to see if truncation is happening a lot

lengths = [len(tokenizer(t, truncation=False)["input_ids"]) for t in df["text"].sample(1000, random_state=42)]
print("p50", np.percentile(lengths, 50))
print("p90", np.percentile(lengths, 90))
print("p99", np.percentile(lengths, 99))

# In this case, we see that 90% of the samples are above 256 tokens, which means truncation is happening for most samples. We might want to consider using a longer max_length or a model that can handle longer inputs if we think important information is being truncated.

Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


p50 410.0
p90 512.1
p99 525.0


In [14]:
# Save the trained model and tokenizer
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.88it/s]


('./model/tokenizer_config.json', './model/tokenizer.json')

In [15]:

# Evaluate and show metrics
metrics = trainer.evaluate(eval_dataset=val_dataset)
print("Validation metrics:", metrics)

# Predict to verify output shape
predictions = trainer.predict(val_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)
print("Sample predictions:", pred_labels[:10])

/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Validation metrics: {'eval_loss': 1.0545731782913208, 'eval_accuracy': 0.4439805149617258, 'eval_runtime': 59.6994, 'eval_samples_per_second': 96.282, 'eval_steps_per_second': 12.044, 'epoch': 2.0}


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Sample predictions: [2 1 0 0 0 2 1 2 0 1]


## 2.Predict

In [16]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Using device:", device)

# model = AutoModelForSequenceClassification.from_pretrained(model_save_path).to(device)
# tokenizer = AutoTokenizer.from_pretrained(model_save_path)

# model.eval()

In [17]:
# 0) Load test data
test_df = pd.read_csv(test_data_path)


# 3) Build text field
if set(["prompt", "response_a", "response_b"]).issubset(test_df.columns):
    test_df["text"] = test_df.apply(build_text, axis=1)
else:
    raise ValueError("Missing one of prompt/response_a/response_b")

# 4) HF dataset
test_dataset = Dataset.from_pandas(test_df[["text"]])

# 6) tokenize
test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["input_ids", "attention_mask", "text"]])



Map: 100%|██████████| 3/3 [00:00<00:00, 1482.78 examples/s]


In [18]:


# 4) Predict
preds = trainer.predict(test_dataset)
print("Raw predictions shape:", preds)
logits = preds.predictions  # shape (N, 3)
print("Logits shape:", logits)
logits_tensor = torch.from_numpy(logits).float()

if torch.isnan(logits_tensor).any() or torch.isinf(logits_tensor).any():
    raise ValueError("Logits contain NaN or Inf; check model/prediction pipeline")

# stable softmax to avoid numeric instabilities
logits_stable = logits_tensor - logits_tensor.max(dim=-1, keepdim=True).values
exp_scores = torch.exp(logits_stable)
probs_tensor = exp_scores / exp_scores.sum(dim=-1, keepdim=True)
probs = probs_tensor.cpu().numpy()

# 5) save to submission file
out = pd.DataFrame({
    "id": test_df["id"].values,
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_model_tie": probs[:, 2],
})

out.to_csv(data_submission_path, index=False)
print(out.head())


Raw predictions shape: PredictionOutput(predictions=array([[-0.13840683, -0.17017023,  0.26784518],
       [ 0.5221141 , -0.59545743, -0.03855899],
       [-0.23560141,  0.44480225, -0.35040006]], dtype=float32), label_ids=None, metrics={'test_runtime': 0.0625, 'test_samples_per_second': 48.029, 'test_steps_per_second': 16.01})
Logits shape: [[-0.13840683 -0.17017023  0.26784518]
 [ 0.5221141  -0.59545743 -0.03855899]
 [-0.23560141  0.44480225 -0.35040006]]
        id  winner_model_a  winner_model_b  winner_model_tie
0   136060        0.288191        0.279181          0.432627
1   211333        0.526899        0.172334          0.300767
2  1233961        0.258651        0.510751          0.230599


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
